# SQL-LoRA: Fine-Tune Llama-3.2-3B for Text-to-SQL

Fine-tunes a 4-bit quantized LLM with QLoRA on `b-mc2/sql-create-context`.
Designed for **Google Colab free tier (T4 GPU, 15 GB VRAM)**.

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `data/train.jsonl` from this repo to the Colab runtime (or re-download it)
3. Run all cells

## Output
- LoRA adapter (~100 MB) pushed to Hugging Face Hub
- Training loss plot
- Quick inference test

---
## 1. Install Dependencies

In [ ]:
%%capture
    import os, re# Unsloth's recommended Colab install (auto-detects T4 vs newer GPUs)if "COLAB_" not in "".join(os.environ.keys()):    !pip install unslothelse:    import torch    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)    xformers = "xformers==" + {"2.10":"0.0.34","2.9":"0.0.33.post1","2.8":"0.0.32.post2"}.get(v, "0.0.34")    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth    !pip install --no-deps --upgrade "torchao>=0.16.0"!pip install transformers==4.56.2!pip install --no-deps trl==0.22.2print("Dependencies installed.")

---
## 2. Imports & Configuration

In [ ]:
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from huggingface_hub import notebook_login
from transformers import DataCollatorForSeq2Seq
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
# ── Paths ──────────────────────────────────────────────────────────────# In Colab, upload train.jsonl or clone the repoTRAIN_PATH = "train.jsonl"  # update if uploaded elsewhere# ── Model ──────────────────────────────────────────────────────────────BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"MAX_SEQ_LENGTH = 512# ── LoRA hyperparameters ──────────────────────────────────────────────LORA_R = 16LORA_ALPHA = 16  # best practice: alpha = rLORA_DROPOUT = 0.0TARGET_MODULES = [    "q_proj", "k_proj", "v_proj", "o_proj",    "gate_proj", "up_proj", "down_proj",]# ── Training hyperparameters ──────────────────────────────────────────PER_DEVICE_BATCH_SIZE = 2GRADIENT_ACCUMULATION_STEPS = 4# effective batch size = 2 * 4 = 8LEARNING_RATE = 2e-4WARMUP_STEPS = 5MAX_STEPS = 200  # ~2-3 epochs over 1200 examples at effective batch 8LOGGING_STEPS = 10SAVE_STEPS = 50OUTPUT_DIR = "checkpoints"# ── Hub ───────────────────────────────────────────────────────────────# Set this to your HF Hub repo (e.g. "your-username/sql-lora-llama3.2-3b")HF_ADAPTER_REPO = ""  # <-- FILL THIS IN BEFORE RUNNINGprint("Configuration loaded.")

---
## 3. Load & Format Dataset

We load the prepared `train.jsonl` (1200 examples, stratified by complexity)
and format each example into the Llama 3.1 chat template:
- **User turn**: schema + question
- **Assistant turn**: gold SQL query

In [ ]:
# ── Load JSONL ─────────────────────────────────────────────────────────rows = []with open(TRAIN_PATH, encoding="utf-8") as f:    for line in f:        rows.append(json.loads(line))print(f"Loaded {len(rows)} training examples")# Previewprint(f"\nExample:\n  Q: {rows[0]['question'][:70]}")print(f"  SQL: {rows[0]['answer'][:70]}")print(f"  Schema: {rows[0]['context'][:70]}")

In [ ]:
# ── Convert to HF Dataset ──────────────────────────────────────────────dataset = Dataset.from_list(rows)dataset = dataset.train_test_split(test_size=0, seed=42)["train"]print(f"Dataset size: {len(dataset)}")

In [ ]:
# ── Format into chat template ──────────────────────────────────────────def format_sql_prompt(examples):    """Convert (context, question, answer) into Llama 3.1 chat format."""    texts = []    for ctx, q, a in zip(examples["context"], examples["question"], examples["answer"]):        messages = [            {                "role": "user",                "content": f"{ctx}\n\nQuestion: {q}",            },            {"role": "assistant", "content": a},        ]        text = tokenizer.apply_chat_template(            messages, tokenize=False, add_generation_prompt=False        )        texts.append(text)    return {"text": texts}dataset = dataset.map(format_sql_prompt, batched=True)print(f"Formatted example:\n{dataset[0]['text'][:300]}\n...")

---
## 4. Load Base Model (4-bit)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(    model_name=BASE_MODEL,    max_seq_length=MAX_SEQ_LENGTH,    dtype=None,  # auto-detect (Float16 for T4, BFloat16 for Ampere+)    load_in_4bit=True,)print(f"Model loaded: {BASE_MODEL}")

---
## 5. Set Chat Template

In [ ]:
tokenizer = get_chat_template(    tokenizer,    chat_template="llama-3.1",  # Llama 3.2 uses Llama 3.1 format)print("Chat template set to llama-3.1")

---
## 6. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r=LORA_R,    target_modules=TARGET_MODULES,    lora_alpha=LORA_ALPHA,    lora_dropout=LORA_DROPOUT,    bias="none",    use_gradient_checkpointing="unsloth",    random_state=3407,    use_rslora=False,    loftq_config=None,)print(f"LoRA adapters added. Trainable params: {model.get_nb_trainable_parameters()[0]:,}")

---
## 7. Configure & Run Training

In [ ]:
# @title Show GPU stats before traininggpu_stats = torch.cuda.get_device_properties(0)start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")print(f"{start_gpu_memory} GB reserved before training.")

In [ ]:
trainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=dataset,    dataset_text_field="text",    max_seq_length=MAX_SEQ_LENGTH,    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),    packing=False,    args=SFTConfig(        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,        warmup_steps=WARMUP_STEPS,        max_steps=MAX_STEPS,        learning_rate=LEARNING_RATE,        logging_steps=LOGGING_STEPS,        save_steps=SAVE_STEPS,        optim="adamw_8bit",        weight_decay=0.001,        lr_scheduler_type="linear",        seed=3407,        output_dir=OUTPUT_DIR,        fp16=not is_bfloat16_supported(),        bf16=is_bfloat16_supported(),        report_to="none",    ),)print("Trainer configured.")

In [ ]:
# ── Mask loss on user turns (train only assistant responses) ────────────trainer = train_on_responses_only(    trainer,    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",)print("Loss masking configured.")

In [ ]:
# ── Train ───────────────────────────────────────────────────────────────trainer_stats = trainer.train()print(f"\nTraining complete in {trainer_stats.metrics['train_runtime']:.1f}s")

In [ ]:
# @title Show final GPU statsused_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)used_memory_for_lora = round(used_memory - start_gpu_memory, 3)max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)print(f"Peak reserved memory = {used_memory} GB.")print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")print(f"Peak reserved memory % of max memory = {round(used_memory / max_memory * 100, 1)} %.")print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s "      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")

---
## 8. Training Loss Plot

In [ ]:
# ── Extract loss from training logs ─────────────────────────────────────log_history = trainer.state.log_historysteps = []losses = []for entry in log_history:    if "loss" in entry:        steps.append(entry["step"])        losses.append(entry["loss"])plt.figure(figsize=(10, 4))plt.plot(steps, losses, marker="o", markersize=3, linewidth=1.5)plt.xlabel("Training Step")plt.ylabel("Loss")plt.title("SQL-LoRA Training Loss")plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()if losses:    print(f"Initial loss: {losses[0]:.4f}")    print(f"Final loss:   {losses[-1]:.4f}")    print(f"Reduction:    {losses[0] - losses[-1]:.4f}")

---
## 9. Save Adapter & Push to Hugging Face Hub

The adapter file is tiny (~100 MB). The base model weights stay on HF.

1. Login to HF Hub: run the cell below and paste your token
2. Set `HF_ADAPTER_REPO` to `"your-username/sql-lora-llama3.2-3b"`
3. Run the push cell

In [ ]:
notebook_login()

In [ ]:
if HF_ADAPTER_REPO:    print(f"Pushing adapter to {HF_ADAPTER_REPO}...")    model.push_to_hub_merged(        HF_ADAPTER_REPO,        tokenizer,        save_method="lora",    )    print("Done! Adapter pushed to Hugging Face Hub.")else:    print("HF_ADAPTER_REPO not set. Saving locally instead.")    model.save_pretrained("sql_lora_adapter")    tokenizer.save_pretrained("sql_lora_adapter")    print("Adapter saved to sql_lora_adapter/")

---
## 10. Quick Inference Test

Run a few held-out examples through the fine-tuned model.

In [ ]:
# ── Load a few test examples from the repo's eval set ───────────────────# If eval.jsonl is not available, we use a manual exampletest_examples = [    {        "context": "CREATE TABLE head (age INTEGER, name TEXT)",        "question": "How many heads of the departments are older than 56?",        "answer": "SELECT COUNT(*) FROM head WHERE age > 56",    },    {        "context": "CREATE TABLE department (creation VARCHAR, name VARCHAR, budget_in_billions VARCHAR)",        "question": "List the creation year, name and budget of each department.",        "answer": "SELECT creation, name, budget_in_billions FROM department",    },    {        "context": "CREATE TABLE trip (duration INTEGER, start_station_id VARCHAR) CREATE TABLE station (name VARCHAR, id VARCHAR)",        "question": "What are the names of stations that had more than 14 bikes available?",        "answer": "SELECT s.name FROM station s WHERE s.id IN (SELECT t.start_station_id FROM trip t)",    },]# ── Inference ──────────────────────────────────────────────────────────FastLanguageModel.for_inference(model)for ex in test_examples:    messages = [        {"role": "user", "content": f"{ex['context']}\n\nQuestion: {ex['question']}"},    ]    inputs = tokenizer.apply_chat_template(        messages,        tokenize=True,        add_generation_prompt=True,        return_tensors="pt",    ).to("cuda")    outputs = model.generate(        input_ids=inputs,        max_new_tokens=256,        temperature=0.1,        top_p=0.95,        use_cache=True,    )    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()    print(f"\nQ: {ex['question']}")    print(f"Gold: {ex['answer']}")    print(f"Pred: {response}")    print(f"Match: {'✓' if response.strip().upper().startswith(ex['answer'].strip().upper()[:20]) else '?'}")

---
## Next Steps

1. Push the adapter to HF Hub (Cell 9)  
2. Run `run_finetuned_eval.py` locally with `--adapter your-username/sql-lora-llama3.2-3b`  
3. Compare results against the baseline in `results/baseline_results.json`

See the project README for full reproduction instructions.